In [0]:
import sys
from pyspark.sql.functions import *
import tomllib
from functools import reduce
# Append the path to the repo
user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")

In [0]:
config_path = f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting/config.toml"
with open(config_path, "rb") as f:
    config = tomllib.load(f)

validation = config["validation"]

temp_min = validation["temperature"]["min"]
temp_max = validation["temperature"]["max"]
wind_min = validation["wind_speed"]["min"]                
wind_max = validation["wind_speed"]["max"]
rad_min  = validation["radiation"]["min"]
rad_max  = validation["radiation"]["max"]


In [0]:
# Weather data is hourly but we need it to be half hourly to match the settlement periods of energy data


def _create_half_hourly(key, df):
    "Pandas function to create half hourly data from hourly data"
    df = df.set_index("timestamp")
    df = df.resample("30min").ffill()
    df = df.reset_index()    
    return df




In [0]:
# Read the weather data in from Bronze layer

bronze_weather = spark.read.table("bronze_weather_regional")

In [0]:

schema = "region_id long, region_name string, timestamp timestamp, temperature double, wind_speed double, radiation double" 
silver_weather = bronze_weather.groupBy("region_id").applyInPandas(_create_half_hourly, schema=schema)

In [0]:
display(silver_weather.limit(25))

In [0]:
display(silver_weather.describe())

Noticing that I am 14 rows short of the expected (2024 is a leap year, 366 days × 48) = 245,952 rows. That's exactly 1 missing half-hour per region, likely the very last slot at the end of the year (2024-12-31T23:30:00) not being generated because resample stops at the last observed timestamp.


In [0]:
# Validate the temp, wind and solar
silver_weather = silver_weather.withColumn("wind_valid", (col("wind_speed") >= wind_min) & (col("wind_speed") <= wind_max))\
                .withColumn("temp_valid", (col("temperature") >= temp_min) & (col("temperature") <= temp_max))\
                .withColumn("rad_valid", (col("radiation") >= rad_min) & (col("radiation") <= rad_max))

display(silver_weather.limit(25))              

In [0]:
# Create a has_nulls column that checks for nulls in the key columns, wind and temp
key_cols = ["temperature", "wind_speed", "radiation"]
has_nulls = [col(name).isNull() for name in key_cols]

silver_weather = silver_weather.withColumn("has_nulls", reduce(lambda a, b: a | b, has_nulls))

In [0]:

display(silver_weather.filter(col("has_nulls")).limit(25))   

In [0]:
# Write silver layer table with merge upsert 

# create temp view
silver_weather.createOrReplaceTempView("weather_staging_silver")

# create table if it doesn't exist
spark.sql("CREATE TABLE IF NOT EXISTS silver_weather USING DELTA AS SELECT * FROM weather_staging_silver WHERE 1=0")

# Use MERGE INTO statement to "upsert" the data into the silver table
spark.sql("""
          MERGE INTO silver_weather
          USING weather_staging_silver
          ON silver_weather.region_id = weather_staging_silver.region_id
          AND silver_weather.timestamp = weather_staging_silver.timestamp
          WHEN NOT MATCHED THEN INSERT *""")


In [0]:
display(silver_weather.describe())